In [0]:
# ================================================================
# NOTEBOOK: nb_gold_rfm_segments
# PURPOSE:  Customer RFM Segmentation
# RUN:      Daily (after nb_gold_daily_sales completes)
# READS:    silver/orders + silver/customers
# WRITES:   gold/rfm_segments/
# GRAIN:    1 row per Customer (current snapshot)
#
# WHAT IS RFM?
#   R = Recency   → How recently did customer last order?
#                   Recent buyer = high R score
#   F = Frequency → How many orders has customer placed?
#                   More orders = high F score
#   M = Monetary  → How much total has customer spent?
#                   High spender = high M score
#
# WHY RFM MATTERS FOR SHOPSENSE?
#   Marketing team uses RFM to target campaigns:
#   CHAMPIONS      → Best customers, reward them
#   LOYAL          → Buy often, upsell to premium
#   AT_RISK        → Used to buy, now quiet, re-engage
#   LOST           → Haven't bought in months, win-back campaign
#   NEW            → Just joined, nurture them
# ================================================================


from pyspark.sql import functions as F
from pyspark.sql.functions import (col, datediff, current_date, round, when, lit, count, sum, max, min, percent_rank, ntile) 
from pyspark.sql.window import Window


STORAGE = ("abfss://source@stshopsensedevhj.dfs.core.windows.net")

ORDERS_PATH = f"{STORAGE}/silver/orders/"
CUSTOMERS_PATH = f"{STORAGE}/silver/customers/"
GOLD_PATH = f"{STORAGE}/gold/rfm_segments/"


# ================================================================
# STEP 1: READ SILVER TABLES
# ================================================================


orders = (
    spark.read
    .format("delta")
    .load(ORDERS_PATH)
)


customers = (
    spark.read
    .format("delta")
    .load(CUSTOMERS_PATH)
)

# Keep only active orders if the soft-delete column exists
if "_is_deleted" in orders.columns:
    orders = orders.filter(F.coalesce(F.col("_is_deleted"),F.lit(False)) == False
)


# Keep only current customer versions
if "is_current" in customers.columns:
    customers = customers.filter(
        col("is_current") == True
    )


# Keep only active customers if the soft-delete column exists
if "_is_deleted" in customers.columns:
    customers = filter(coalesce(F.col("_is_deleted"), F.lit(False)) == False
)


customers = (
    customers
    .select(
        "CustomerID",
        "FullName",
        "City",
        "State",
        "Segment",
        "IsPrimeBool",
        "TenureCategory",
        "IsHighValue"
    )
)


print(
    f"[READ] Orders: {orders.count()} | "
    f"Current customers: {customers.count()}"
)


# ================================================================
# STEP 2: VALIDATE REQUIRED ORDER COLUMNS
# ================================================================

required_order_columns = [
    "OrderID",
    "CustomerID",
    "OrderDate",
    "NetAmount",
    "IsDelivered",
    "IsCancelled",
    "IsPrimeOrder"
]

missing_order_columns = []

for column_name in required_order_columns:
    if column_name not in orders.columns:
        missing_order_columns.append(column_name)

    
if len(missing_order_columns) > 0:
    missing_names = ", ".join(missing_order_columns)


    raise ValueError(
        "Missing required columns in silver/orders: "               #Convert the missing-column list into readable text, then stop the notebook and display which columns are missing.
        + missing_names
)



# ================================================================
# STEP 3: PREPARE ORDER DATA
# ================================================================

orders_prepared = (
    orders
    .withColumn("OrderDateOnly", F.to_date(F.col("OrderDate")))
    .withColumn("NetAmount",   F.coalesce(F.col("NetAmount").cast("double"),F.lit(0.0)))
    .withColumn(
        "IsDelivered",
        F.coalesce(
            F.col("IsDelivered").cast("boolean"),
            F.lit(False)
        )
    )
    .withColumn(
        "IsCancelled",
        F.coalesce(
            F.col("IsCancelled").cast("boolean"),
            F.lit(False)
        )
    )
    .withColumn(
        "IsPrimeOrder",
        F.coalesce(
            F.col("IsPrimeOrder").cast("boolean"),
            F.lit(False)
        )
    )
)



# ================================================================
# STEP 4: FILTER DELIVERED ORDERS
# ================================================================

# RFM revenue and frequency should use successfully delivered orders

delivered = (
    orders_prepared
    .filter(col("IsDelivered")  == True)
    .filter(col("CustomerID").isNotNull())
    .filter(col("OrderID").isNotNull())
    .filter(F.col("OrderDateOnly").isNotNull())
)

delivered_count = delivered.count()


print(f"[FILTER] Delivered orders: {delivered_count}")

if delivered_count == 0:
    raise ValueError(
        "No delivered orders found. "
        "RFM segmentation cannot be calculated."
    )




# ================================================================
# STEP 5: DETERMINE THE ANALYSIS DATE
# ================================================================

analysis_date = (
    delivered
    .agg(
        max("OrderDateOnly").alias("AnalysisDate")
    )
    .first()["AnalysisDate"]
)

if analysis_date is None:
    raise ValueError(
        "Analysis date could not be calculated."
    )

print(f"[INFO] RFM analysis date: {analysis_date}")


# ================================================================
# STEP 6: CALCULATE RAW RFM METRICS
# ================================================================

#calculating RFM values for each customer.

# --------------------------------------------------------
 # RECENCY
 # Days between analysis date and latest delivered order.
 #
 # Lower value = customer purchased more recently.
 # --------------------------------------------------------


rfm_raw = (
    delivered
    .groupBy("CustomerID")
    .agg(
        F.datediff(
            F.lit(analysis_date),
            F.max("OrderDateOnly")
            ).alias("Recency_Days"),
        F.countDistinct(
            "OrderID"
        ).alias("Frequency"),
        F.round(
            F.sum("NetAmount"),
            2
        ).alias("Monetary"),


        # Average delivered order value
        F.round(
            F.avg("NetAmount"),
            2
        ).alias("AvgOrderValue"),


        # First delivered order date
        F.min(
            "OrderDateOnly"
        ).alias("FirstOrderDate"),


        # Most recent delivered order date
        F.max(
            "OrderDateOnly"
        ).alias("LastOrderDate"),


        # Number of unique Prime orders
        F.countDistinct(
            F.when(
                F.col("IsPrimeOrder") == True,
                F.col("OrderID")
            )
        ).alias("PrimeOrders")
    )
)



# ================================================================
# STEP 7: CALCULATE ALL-ORDER AND CANCELLATION STATISTICS
# ================================================================

customer_order_stats = (
    orders_prepared
    .filter(F.col("CustomerID").isNotNull())
    .filter(F.col("OrderID").isNotNull())
    .groupBy("CustomerID")
    .agg(
        F.countDistinct("OrderID").alias("AllOrders"),

        F.countDistinct(
            F.when(
                F.col("IsCancelled") == True,
                F.col("OrderID")
            )
        ).alias("CancelledOrders")
    )
)
rfm_raw = (
    rfm_raw
    .join(customer_order_stats,                # Join with customer_order_stats
          on = "CustomerID",
          how = "left"
          )
    .fillna({"AllOrders": 0,
             "CancelledOrders": 0
             }
        )
)
    

 # ================================================================
# STEP 8: SCORE R, F AND M ON A 1–5 SCALE
# ================================================================

# ntile(5) divides customers into five approximately equal groups.
#
# RECENCY:
#   High Recency_Days = customer ordered a long time ago = score 1
#   Low Recency_Days  = customer ordered recently        = score 5
#
# FREQUENCY:
#   Low Frequency  = score 1
#   High Frequency = score 5
#
# MONETARY:
#   Low Monetary  = score 1
#   High Monetary = score 5


# Descending recency:
# Old customers come first and receive lower ntile scores.   

# Descending recency:
w_recency = Window.orderBy(F.col("Recency_Days").desc()
)


# Ascending frequency:
# # Low-frequency customers receive lower scores
w_frequency = Window.orderBy(F.col("Frequency").asc()
)  



# Ascending monetary:
# Low-spending customers receive lower scores.

w_monetary = Window.orderBy(F.col("Monetary").asc()
)


rfm_scored = (
    rfm_raw
    .withColumn("R_Score",  F.ntile(5).over(w_recency)
    )

    .withColumn(
        "F_Score",
        F.ntile(5).over(w_frequency)
    )

.withColumn("M_Score", F.ntile(5).over(w_monetary))
)

# ================================================================
# STEP 9: CREATE COMPOSITE RFM SCORES
# ================================================================


rfm_scored = (
    rfm_scored

    # Simple score:
    # Minimum = 3
    # Maximum = 15
    .withColumn(
        "RFM_Score",
        F.col("R_Score")
        + F.col("F_Score")
        + F.col("M_Score")
    )
    # Weighted score:
    # Recency   = 40%
    # Frequency = 35%
    # Monetary  = 25%
    .withColumn(
        "RFM_Weighted",
        F.round(
            F.col("R_Score") * 0.40
            + F.col("F_Score") * 0.35
            + F.col("M_Score") * 0.25,
            2
        )
    )

    # Optional readable three-digit RFM code
    # Example: R=5, F=4, M=3 becomes "543"

    .withColumn(
        "RFM_Code",
        F.concat(
            F.col("R_Score").cast("string"),
            F.col("F_Score").cast("string"),
            F.col("M_Score").cast("string")
        )
    )
)

# ================================================================
# STEP 10: ASSIGN CUSTOMER SEGMENT LABELS
# ================================================================

# Important:
# Conditions are checked from top to bottom.

rfm_segmented = (
    rfm_scored
    .withColumn(
        "CustomerSegment",

        F.when(
            (F.col("R_Score") >= 4) &
            (F.col("F_Score") >= 4) &
            (F.col("M_Score") >= 4),
            "CHAMPIONS"
        )

        .when(
            (F.col("F_Score") >= 4) &
            (F.col("R_Score") >= 3),
            "LOYAL"
        )

        .when(
            (F.col("R_Score") >= 4) &
            (F.col("Frequency") == 1),
            "NEW_CUSTOMERS"
        )

        .when(
            (F.col("R_Score") >= 4) &
            (F.col("Frequency") > 1) &
            (F.col("F_Score").between(2, 3)),
            "POTENTIAL_LOYALISTS"
        )

        .when(
            (F.col("R_Score") == 3) &
            (F.col("F_Score") <= 2),
            "PROMISING"
        )

        .when(
            (F.col("R_Score") == 3) &
            (F.col("F_Score").between(3, 4)),
            "NEED_ATTENTION"
        )

        .when(
            (F.col("R_Score") <= 2) &
            (F.col("F_Score") >= 4) &
            (F.col("M_Score") >= 4),
            "CANT_LOSE_THEM"
        )

        .when(
            (F.col("R_Score") <= 2) &
            (F.col("F_Score") >= 3),
            "AT_RISK"
        )

        .when(
            (F.col("R_Score") == 1) &
            (F.col("F_Score") == 1) &
            (F.col("M_Score") <= 2),
            "LOST"
        )

        .when(
            (F.col("R_Score") <= 2) &
            (F.col("F_Score") <= 2) &
            (F.col("M_Score") <= 2),
            "HIBERNATING"
        )

        .when(
            (F.col("R_Score").between(2, 3)) &
            (F.col("F_Score") <= 2),
            "ABOUT_TO_SLEEP"
        )

        .otherwise("OTHERS")
    )
)

# ================================================================
# STEP 11: ADD VALUE AND BEHAVIOUR COLUMNS
# ================================================================

rfm_segmented = (
    rfm_segmented

    # Customer value based on lifetime delivered revenue
    .withColumn(
        "ValueTier",

        F.when(
            F.col("Monetary") >= 50000,
            "PLATINUM"
        )

        .when(
            F.col("Monetary") >= 20000,
            "GOLD"
        )

        .when(
            F.col("Monetary") >= 5000,
            "SILVER"
        )

        .otherwise("BRONZE")
    )
    # Percentage of all orders that were cancelled

    .withColumn("CancellationRate",
           when(
               col("AllOrders") > 0,
           F.round(col("CancelledOrders")
                 / col("AllOrders")
                 * 100,
                 2
           )
    )
    .otherwise(F.lit(0.0))
    )
    # Number of days between first and most recent delivered order
    .withColumn(
        "DaysBetweenFirstAndLast",
        F.datediff(
            F.col("LastOrderDate"),
            F.col("FirstOrderDate")
        )
    )


# Percentage of delivered orders that were Prime orders  
    .withColumn("PrimeOrderPct",
                F.when(
                    col("Frequency") > 0,
                    round(
                        col("PrimeOrders") / col("Frequency") * 100,
                        2
                    )
                ).otherwise(F.lit(0.0))
            )
)
    
# ================================================================
# STEP 12: JOIN CURRENT CUSTOMER PROFILE
# ================================================================

gold_df = (
    rfm_segmented
    .join(customers,
          on = "CustomerID",
          how = "left"
        )
    
    .withColumn("_analysis_date",
                F.lit(analysis_date)
                )
    .withColumn("_gold_load_ts",
                F.current_timestamp()
    )
)
# ================================================================
# STEP 13: SELECT FINAL COLUMN ORDER
# ================================================================


gold_df = (
    gold_df
    .select(
     # Customer information
        "CustomerID",
        "FullName",
        "City",
        "State",
        "Segment",
        "IsPrimeBool",
        "TenureCategory",
        "IsHighValue",  
        # Segment results
        "CustomerSegment",     
        "ValueTier",
        # Raw RFM metrics
        "R_Score",
        "F_Score",
        "M_Score",
        "RFM_Code",
        "RFM_Score",
        "RFM_Weighted",
        "Recency_Days",
        "Frequency",
        "Monetary",
        "AvgOrderValue",
        # Order behaviour
        "AllOrders",
        "PrimeOrders",
        "PrimeOrderPct",
        "CancelledOrders",
        "CancellationRate",
        "FirstOrderDate",
        "LastOrderDate",
        "DaysBetweenFirstAndLast",
        # Technical metadata
        "_analysis_date",
        "_gold_load_ts"
    )
)

# ================================================================
# STEP 14: WRITE GOLD DELTA TABLE
# ================================================================
(
gold_df.write
.format("delta")
.mode("overwrite")
.option("overwriteSchema", "true")
.save(GOLD_PATH)
)


# ================================================================
# STEP 15: VALIDATION
# ================================================================

total_customers = gold_df.count()

total_revenue = (
    gold_df
    .agg(
        sum("Monetary").alias("TotalRevenue")
    )
    .first()["TotalRevenue"]
)

total_revenue = total_revenue or 0.0


print(
    f"\n[DONE] gold/rfm_segments/ written: "
    f"{total_customers} rows"
)

print(
    f"[VALIDATION] Total customer revenue: "
    f"₹{total_revenue:,.2f}"
)

print(
    f"[VALIDATION] Analysis date: {analysis_date}"
)



# ================================================================
# RFM SEGMENT BREAKDOWN
# ================================================================

print("\n[RFM SEGMENT BREAKDOWN]")


(
gold_df
.groupBy(
    "CustomerSegment"     
)
.agg(
        F.countDistinct("CustomerID").alias("CustomerCount"),
        F.round(F.sum("Monetary"), 2).alias("TotalRevenue"),
        F.round(F.avg("Monetary"), 2).alias("AvgRevenue"),
        F.round(F.avg("Frequency"), 1).alias("AvgOrders"),
        F.round(F.avg("Recency_Days"), 0).alias("AvgRecencyDays")
    )
    .orderBy(F.col("TotalRevenue").desc())
    .show(20, truncate=False)
)

    
# ================================================================
# VALUE TIER BREAKDOWN
# ================================================================

print("\n[VALUE TIER BREAKDOWN]")

(
gold_df
.groupBy(
    "ValueTier"
    )
    .agg(
    F.countDistinct("CustomerID").alias("Customers"),
    F.round(sum("Monetary"),
             2).alias("Revenue")
    )
    .orderBy(F.col("Revenue").desc()
        ).show(truncate=False)
    )



# ================================================================
# DISPLAY TOP CUSTOMERS
# ================================================================

display(
    gold_df
.select(
        "CustomerID",
        "FullName",
        "CustomerSegment",
        "ValueTier",
        "R_Score",
        "F_Score",
        "M_Score",
        "RFM_Code",
        "RFM_Score",
        "RFM_Weighted",
        "Recency_Days",
        "Frequency",
        "Monetary",
        "AvgOrderValue",
        "PrimeOrderPct",
        "CancellationRate",
        "City",
        "Segment",
        "IsPrimeBool"
    )
    .orderBy(
        F.col("RFM_Score").desc(),
        F.col("Monetary").desc()
    )
    .limit(15)
)


[READ] Orders: 3015 | Current customers: 500
[FILTER] Delivered orders: 1453
[INFO] RFM analysis date: 2024-06-29


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(



[DONE] gold/rfm_segments/ written: 477 rows
[VALIDATION] Total customer revenue: ₹36,468,586.14
[VALIDATION] Analysis date: 2024-06-29

[RFM SEGMENT BREAKDOWN]
+-------------------+-------------+-------------+----------+---------+--------------+
|CustomerSegment    |CustomerCount|TotalRevenue |AvgRevenue|AvgOrders|AvgRecencyDays|
+-------------------+-------------+-------------+----------+---------+--------------+
|CHAMPIONS          |94           |1.313996968E7|139786.91 |5.1      |15.0          |
|LOYAL              |63           |5723786.3    |90853.75  |4.1      |25.0          |
|AT_RISK            |51           |3643302.49   |71437.3   |3.2      |85.0          |
|CANT_LOSE_THEM     |26           |3107729.59   |119528.06 |4.4      |82.0          |
|POTENTIAL_LOYALISTS|51           |3038423.0    |59576.92  |2.4      |20.0          |
|NEED_ATTENTION     |26           |1873674.69   |72064.41  |3.0      |42.0          |
|PROMISING          |39           |1727670.01   |44299.23  |1.7  

CustomerID,FullName,CustomerSegment,ValueTier,R_Score,F_Score,M_Score,RFM_Code,RFM_Score,RFM_Weighted,Recency_Days,Frequency,Monetary,AvgOrderValue,PrimeOrderPct,CancellationRate,City,Segment,IsPrimeBool
CUST00016,Sneha Reddy,CHAMPIONS,PLATINUM,5,5,5,555,15,5.0,1,7,280448.67,40064.1,42.86,12.5,Pune,REGULAR,false
CUST00328,Sneha Kumar,CHAMPIONS,PLATINUM,5,5,5,555,15,5.0,7,7,247154.6,35307.8,28.57,25.0,Ahmedabad,REGULAR,true
CUST00490,Arjun Joshi,CHAMPIONS,PLATINUM,5,5,5,555,15,5.0,0,8,239200.43,29900.05,50.0,16.67,Kolkata,REGULAR,true
CUST00106,Meena Nair,CHAMPIONS,PLATINUM,5,5,5,555,15,5.0,7,9,231511.35,25723.48,22.22,15.38,Jaipur,VIP,true
CUST00055,Amit Agarwal,CHAMPIONS,PLATINUM,5,5,5,555,15,5.0,3,7,200815.88,28687.98,28.57,0.0,Mumbai,PREMIUM,false
CUST00082,Kavya Reddy,CHAMPIONS,PLATINUM,5,5,5,555,15,5.0,5,6,195099.15,32516.53,33.33,0.0,Hyderabad,VIP,true
CUST00260,Priya Sharma,CHAMPIONS,PLATINUM,5,5,5,555,15,5.0,1,8,193313.93,24164.24,75.0,0.0,Chennai,VIP,true
CUST00192,Amit Kumar,CHAMPIONS,PLATINUM,5,5,5,555,15,5.0,13,5,192884.52,38576.9,60.0,0.0,Hyderabad,PREMIUM,false
CUST00329,Sneha Reddy,CHAMPIONS,PLATINUM,5,5,5,555,15,5.0,7,8,189833.71,23729.21,75.0,0.0,Pune,REGULAR,true
CUST00095,Meena Sharma,CHAMPIONS,PLATINUM,5,5,5,555,15,5.0,3,7,189532.18,27076.03,85.71,10.0,Pune,REGULAR,true


In [0]:
customer_order_stats.printSchema()

root
 |-- CustomerID: string (nullable = true)
 |-- AllOrders: long (nullable = false)
 |-- count(DISTINCT CASE WHEN (IsCancelled = true) THEN OrderID END AS CancelledOrders): long (nullable = false)



In [0]:
print(rfm_segmented.columns)

['CustomerID', 'Recency_Days', 'Frequency', 'Monetary', 'AvgOrderValue', 'FirstOrderDate', 'LastOrderDate', 'PrimeOrders', 'AllOrders', 'CancelledOrders', 'R_Score', 'F_Score', 'M_Score', 'RFM_Score', 'RFM_Weighted', 'RFM_Code', 'CustomerSegment', 'ValueTier', 'CancellationRate', 'PrimeOrderPct']
